<a href="https://colab.research.google.com/github/kostismatz/GKS_ML_Project/blob/main/NPL_2_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [7]:
import torch
print(torch.__version__)



2.10.0+cu128


In [20]:
# -*- coding: utf-8 -*-
"""

A RNN classifier applied to AG_NEWS dataset

Download dataset:
https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset

"""

import torch
import time
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import functional as F
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

MAX_WORDS = 25
EPOCHS = 15
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EMBEDDING_DIM = 100
HIDDEN_DIM = 64

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')


cuda


In [21]:
######################################################################
# Data processing
# -----------------------------


def tokenizer(text):
    return text.lower().split()

# All texts are truncated and padded to MAX_WORDS tokens
def collate_batch(batch):
    Y, X = list(zip(*batch))
    Y = torch.tensor(Y) - 1 # Target names in range [0,1,2,3] instead of [1,2,3,4]
    X = [[vocab.get(token, vocab["<UNK>"]) for token in tokenizer(text)]for text in X]
    # Bringing all samples to MAX_WORDS length. Shorter texts are padded with <PAD> sequences, longer texts are truncated.
    X = [tokens + [vocab["<PAD>"]] * (MAX_WORDS - len(tokens))
    if len(tokens) < MAX_WORDS
    else tokens[:MAX_WORDS]
    for tokens in X]
    return torch.tensor(X, dtype=torch.long).to(device), Y.to(device)

train_dataset = [(label,train_data['Title'][i] + ' ' + train_data['Description'][i]) for i,label in enumerate(train_data['Class Index'])]
test_dataset = [(label,test_data['Title'][i] + ' ' + test_data['Description'][i]) for i,label in enumerate(test_data['Class Index'])]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_batch)

target_classes = ["World", "Sports", "Business", "Sci/Tech"]

def build_vocabulary(dataset, min_freq=10):
    counter = Counter()

    for _, text in dataset:
        tokens = tokenizer(text)
        counter.update(tokens)

    # Φτιάχνουμε λεξικό manually
    vocab_dict = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    idx = 2
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab_dict[token] = idx
            idx += 1

    return vocab_dict

vocab = build_vocabulary(train_dataset, min_freq=10)

######################################################################

In [22]:
# Define the model
# ----------------


class model(nn.Module):
    def __init__(self,input_dim, embedding_dim, hidden_dim, output_dim):
        super(model, self).__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=input_dim, embedding_dim=embedding_dim)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, X_batch):
        embeddings = self.embedding_layer(X_batch)
        output, hidden = self.rnn(embeddings)
        logits = self.linear(output[:, -1])
        return logits

######################################################################

In [23]:
# Initiate an instance of the model
# ---------------------------------


classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)
# Define loss function and opimization algorithm
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([param for param in classifier.parameters() if param.requires_grad == True],lr=LEARNING_RATE)

# Count model parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('\nModel:')
print(classifier)
print('Total parameters: ',count_parameters(classifier))
print('\n\n')

######################################################################


Model:
model(
  (embedding_layer): Embedding(25099, 100)
  (rnn): RNN(100, 64, batch_first=True)
  (linear): Linear(in_features=64, out_features=4, bias=True)
)
Total parameters:  2520784





In [24]:
# Define functions to train and evaluate the model
# ------------------------------------------------


def EvaluateModel(model, loss_fn, val_loader):
    model.eval()
    with torch.no_grad():
        Y_actual, Y_preds, losses = [],[],[]
        for X, Y in val_loader:
            preds = model(X)
            loss = loss_fn(preds, Y)
            losses.append(loss.item())

            Y_actual.append(Y)
            Y_preds.append(preds.argmax(dim=-1))

        Y_actual = torch.cat(Y_actual)
        Y_preds = torch.cat(Y_preds)

    # Returns mean loss, actual labels, predicted labels
    return torch.tensor(losses).mean(), Y_actual.detach().cpu().numpy(), Y_preds.detach().cpu().numpy()


def TrainModel(model, loss_fn, optimizer, train_loader, epochs):
    epoch_times = []

    for i in range(1, epochs+1):
        model.train()
        print('Epoch:', i)

        losses = []

        start_time = time.time()  # ⏱️ START

        for X, Y in tqdm(train_loader):
            Y_preds = model(X)

            loss = loss_fn(Y_preds, Y)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        end_time = time.time()  # ⏱️ END

        epoch_time = end_time - start_time
        epoch_times.append(epoch_time)

        print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
        print("Epoch Time (sec): {:.2f}".format(epoch_time))

    # επιστροφή μέσου χρόνου
    print("\nAverage Epoch Time:", sum(epoch_times)/len(epoch_times))

TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)

######################################################################

Epoch: 1


100%|██████████| 118/118 [00:01<00:00, 62.99it/s]


Train Loss : 1.260
Epoch Time (sec): 1.88
Epoch: 2


100%|██████████| 118/118 [00:02<00:00, 42.88it/s]


Train Loss : 0.755
Epoch Time (sec): 2.76
Epoch: 3


...

Train Loss : 0.169
Epoch Time (sec): 2.64
Epoch: 14


100%|██████████| 118/118 [00:02<00:00, 49.62it/s]


Train Loss : 0.152
Epoch Time (sec): 2.38
Epoch: 15


100%|██████████| 118/118 [00:02<00:00, 57.64it/s]

Train Loss : 0.143
Epoch Time (sec): 2.05

Average Epoch Time: 2.148950131734212


In [25]:
# Evaluate the model with test dataset
# ------------------------------------


_, Y_actual, Y_preds = EvaluateModel(classifier, loss_fn, test_loader)

print("\nTest Accuracy : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
print("\nClassification Report : ")
print(classification_report(Y_actual, Y_preds, target_names=target_classes))
print("\nConfusion Matrix : ")
print(confusion_matrix(Y_actual, Y_preds))


Test Accuracy : 0.877

Classification Report : 
              precision    recall  f1-score   support

       World       0.89      0.87      0.88      1900
      Sports       0.95      0.92      0.93      1900
    Business       0.82      0.86      0.84      1900
    Sci/Tech       0.85      0.86      0.85      1900

    accuracy                           0.88      7600
   macro avg       0.88      0.88      0.88      7600
weighted avg       0.88      0.88      0.88      7600


Confusion Matrix : 
[[1660   52  135   53]
 [  71 1742   18   69]
 [  82   11 1631  176]
 [  45   28  198 1629]]


In [26]:
# =========================
# 🔁 3 RUN EXPERIMENT
# =========================

from sklearn.metrics import accuracy_score
import numpy as np

accuracies = []

for run in range(3):
    print(f"\n===== RUN {run+1} =====")

    # νέο model κάθε φορά
    classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)

    optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)

    # training
    TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)

    # evaluation
    _, y_true, y_pred = EvaluateModel(classifier, loss_fn, test_loader)

    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)

    print("Test Accuracy:", acc)


    print("\n===== FINAL RESULTS =====")

mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print("Accuracies:", accuracies)
print("Mean Accuracy:", mean_acc)
print("Std:", std_acc)


===== RUN 1 =====
Epoch: 1


100%|██████████| 118/118 [00:01<00:00, 62.51it/s]


Train Loss : 1.288
Epoch Time (sec): 1.89
Epoch: 2


100%|██████████| 118/118 [00:02<00:00, 57.19it/s]


Train Loss : 0.752
Epoch Time (sec): 2.07
Epoch: 3


...

Train Loss : 0.148
Epoch Time (sec): 2.03
Epoch: 14


100%|██████████| 118/118 [00:01<00:00, 63.51it/s]


Train Loss : 0.132
Epoch Time (sec): 1.86
Epoch: 15


100%|██████████| 118/118 [00:01<00:00, 63.76it/s]


Train Loss : 0.121
Epoch Time (sec): 1.86

Average Epoch Time: 2.041146246592204
Test Accuracy: 0.8802631578947369

===== FINAL RESULTS =====
Accuracies: [0.8901315789473684, 0.8817105263157895, 0.8802631578947369]
Mean Accuracy: 0.8840350877192983
Std: 0.004351177833416081
